In [1]:
!sudo apt-get install zstd

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 42 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 1s (1,104 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package zstd.
(Reading database ... 122354 files and directories currently

In [2]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [3]:
ollama_model_id = "gemma4:e4b"
ollama_embedder_id="qwen3-embedding:4b"
# !pkill -f ollama

In [40]:
import subprocess, os, time, requests

# 1. Start server (use default port 11434 to avoid conflicts)
proc = subprocess.Popen(
    ["ollama", "serve"],
    env={**os.environ, "OLLAMA_HOST": "0.0.0.0:11434", "OLLAMA_ORIGINS": "*"},
    stdout=subprocess.PIPE, stderr=subprocess.PIPE  # don't suppress errors
)

# 2. Wait until ready
for i in range(20):
    try:
        r = requests.get("http://localhost:11434")
        print("✅ Ollama is up!")
        break
    except:
        print(f"⏳ Waiting... ({i+1})")
        time.sleep(2)
else:
    # Print stderr to see what went wrong
    print(proc.stderr.read().decode())
    raise RuntimeError("❌ Ollama failed to start")

⏳ Waiting... (1)
✅ Ollama is up!


In [41]:
!ollama list

NAME                  ID              SIZE      MODIFIED      
qwen3-embedding:4b    df5bd2e3c74c    2.5 GB    6 minutes ago    
gemma4:e4b            c6eb396dbd59    9.6 GB    8 minutes ago    


In [23]:
!ollama pull {ollama_model_id}

In [24]:
!ollama pull {ollama_embedder_id}

In [30]:
%%bash
curl http://localhost:11434/api/chat -d '{
  "model": "gemma4:e4b",
  "stream": false,
  "messages": [
    { "role": "user", "content": "ما عاصمة مصر؟" }
  ]
}'

{"model":"gemma4:e4b","created_at":"2026-04-19T06:07:42.093765236Z","message":{"role":"assistant","content":"عاصمة مصر هي **القاهرة**."},"done":true,"done_reason":"stop","total_duration":202772693591,"load_duration":202066783705,"prompt_eval_count":21,"prompt_eval_duration":91428858,"eval_count":9,"eval_duration":246540756}

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   471  100   342    0   129      1      0  0:05:42  0:03:22  0:02:20    91


In [26]:
!pip install pyngrok

## **Ngrok**

In [38]:
from google.colab import userdata
from pyngrok import ngrok, conf

ngrok_auth = userdata.get('colab-ngrok')

conf.get_default().auth_token = ngrok_auth

port = "11434"

public_url = ngrok.connect(port).public_url
print(public_url)

https://sprinkler-mongoose-tattoo.ngrok-free.dev


In [42]:
%%bash
curl https://sprinkler-mongoose-tattoo.ngrok-free.dev/api/chat -d '{
  "model": "gemma4:e4b",
  "stream": false,
  "messages": [
    { "role": "user", "content": "ما عاصمة السودان؟" }
  ]
}'

{"model":"gemma4:e4b","created_at":"2026-04-19T06:11:17.565762577Z","message":{"role":"assistant","content":"عاصمة السودان هي **الخرطوم**.","thinking":"Thinking Process:\n\n1.  **Analyze the Request:** The user is asking: \"ما عاصمة السودان؟\" (What is the capital of Sudan?).\n2.  **Identify the Information Needed:** The capital city of Sudan.\n3.  **Determine the Answer (Factual Knowledge):** The capital of Sudan is Khartoum.\n4.  **Formulate the Response (in Arabic):** State the capital clearly.\n\n    *   *Self-Correction/Refinement:* While the capital is Khartoum, it is important to be precise. Khartoum is the most commonly accepted answer.\n\n5.  **Final Answer Generation:** عاصمة السودان هي الخرطوم. (The capital of Sudan is Khartoum.)"},"done":true,"done_reason":"stop","total_duration":33160345511,"load_duration":29032029769,"prompt_eval_count":22,"prompt_eval_duration":63210325,"eval_count":165,"eval_duration":3575883329}

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  1136  100   999  100   137     29      4  0:00:34  0:00:33  0:00:01   235
